In [1]:
import zipfile
import pandas as pd
import numpy as np

In [38]:
# read raw data and extract pedestrian count records, if exists
aggregated_data = []
err_processing_data = []
archive = zipfile.ZipFile('data/Miovision_Raw_data_July2024.zip', 'r')

for name in archive.namelist()[1]:
    
    try:
        xlfile = archive.open(name)
        summary = pd.read_excel(xlfile, 'Summary', header = None)
        
        # make sure the study records pedestrian movement
        if summary.iloc[3, 1] == 'All Processed Legs & Movements':
            
            # locate location name, latitude & longitude information
            locname = summary.iloc[8, 1]
            latlon = summary.iloc[9, 1].split(',')
            lat, lon = latlon[0], latlon[1]
            
            # extract pedestrian count information        
            pedestrian_df = pd.read_excel(xlfile, 'Pedestrians', header = 2)
            pedestrian_df['total_Counts'] = pedestrian_df.iloc[:, 1:].sum(axis = 1)
            pedestrian_df['location'] = locname
            pedestrian_df['lat'] = lat
            pedestrian_df['long'] = lon
            
            # put all count records into a table 
            aggregated_data.append(pedestrian_df)
            
    except Exception as e:
        err_processing_data.append([name, e]) # record processing error type for the ease of investigation
        print('Processing error with {}: {}'.format(name.split('/')[1], e))
    
        

Processing error with 100242-SR52andMast.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 100262-SR125andMissionGorgeRoad.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 101843-MissionGorgeRoadandCuyamaca.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 101844-SR52andCuyamaca.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 101846-SR52andCuyamacaEB125Off.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 1062768-Latona-2734-26-2023.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 1062769-Latona-2734-27-2023.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 109356-03-ED-049-15.163-MAY-15-2013.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 124254-SR76andValleyViewSaturdayPeak.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 124452-SR76andValleyViewThursOnly.xlsx: Worksheet named 'Pedestrians' not found
Processing error 

Processing error with 272328-Kin41@BushAMNB.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 272338-BellleHaven@BushAM.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 272371-SBam@Bush.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 273191-WestboundHarvey11-04PM.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 273218-WestboundHarvey11-05-2015.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 281025-MusselShoalsBikePath.xlsx: Excel file format cannot be determined, you must specify an engine manually.
Processing error with 281450-Standard_SCU1RG_2015-11-17_0500.001.xlsx: Excel file format cannot be determined, you must specify an engine manually.
Processing error with 281454-Standard_SCU1RG_2015-12-04_0600.xlsx: Excel file format cannot be determined, you must specify an engine manually.
Processing error with 281465-Standard_SCU1RG_2015-12-04_0600.001.xlsx: Excel file format cannot be determin

Processing error with 519111-POEStudy1B.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 52342-Brennan.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 52343-Brennan.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 52354-brennan.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 52356-Brennan.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 52358-Brennan.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 524495-GopherDay1.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 524498-GopherDay2.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 524502-GopherDay3.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 524505-GopherDay4.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 524510-GopherDay5.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 524522-GopherDay1Loc2.xlsx: Worksheet named 'Pedestr

Processing error with 813684-36&McCoyCounts2-11-2021.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 813688-36&Bowman2-10-2021Counts.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 813689-36&BowmanCounts2-11-2021.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 814555-70&A24Counts2-17-2021.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 814560-70&A24Counts2-18-2021.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 822272-299&CenterStCounts3-23-2021.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 822274-299&CenterStCounts3-24-2021.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 824940-Cressler&299Counts4-1-2021.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 824941-Cressler&299Counts4-2-2021.xlsx: Worksheet named 'Pedestrians' not found
Processing error with 824943-Lincoln&299Counts4-2-2021.xlsx: Worksheet named 'Pedestrians' not fo

In [50]:
# check reading error types
err = pd.DataFrame(err_processing_data).rename(columns = {0: 'filename', 1: 'errortype'})
err['errortype'] = err['errortype'].astype(str)

In [51]:
# 5 files don't have longitude, latitude information, 
# 8 files cannot be open, 
# 270 files don't have pedestrian count sheet
err.groupby('errortype')['filename'].nunique()

errortype
'float' object has no attribute 'split'                                           5
Excel file format cannot be determined, you must specify an engine manually.      8
Worksheet named 'Pedestrians' not found                                         270
Name: filename, dtype: int64

In [134]:
# create count records dataframe
aggregated_df = pd.concat(aggregated_data)

In [135]:
aggregated_df.columns

Index(['Start Time', 'Right', 'Thru', 'Left', 'U-Turn', 'Peds CW', 'Peds CCW',
       'Right.1', 'Thru.1', 'Left.1', 'U-Turn.1', 'Peds CW.1', 'Peds CCW.1',
       'Right.2', 'Thru.2', 'Left.2', 'U-Turn.2', 'Peds CW.2', 'Peds CCW.2',
       'Right.3', 'Thru.3', 'Left.3', 'U-Turn.3', 'Peds CW.3', 'Peds CCW.3',
       'total_Counts', 'location', 'lat', 'long', 'Peds', 'Peds.1', 'Peds.2',
       'Peds.3', 'Hard right', 'Bear right', 'Bear left', 'Peds CW.4',
       'Peds CCW.4', 'Hard left', 'Bear left.1', 'Hard left.1', 'Hard right.1',
       'Bear right.1', 'Right on red', 'Right on red.1', 'Right on red.2',
       'Right on red.3', 'Hard left.2', 'Hard right.2', 'Hard left.3',
       'Hard right.3', 'U-Turn.4', 'Bear left.2', 'Bear right.2',
       'Hard right on red', 'Bear right.3', 'Bear left.3', 'Right.4',
       'Left.4'],
      dtype='object')

In [154]:
ped_df = aggregated_df[['Start Time', 'lat', 'long', 'location', 'total_Counts']]

In [155]:
ped_df.shape

(13088, 5)

In [156]:
ped_df.location.nunique()

554

In [157]:
ped_df['total_Counts'].describe()

count    13088.000000
mean        12.127445
std         26.983737
min          0.000000
25%          0.000000
50%          3.000000
75%         13.000000
max        569.000000
Name: total_Counts, dtype: float64

In [158]:
ped_df.isnull().sum()

Start Time      0
lat             0
long            0
location        0
total_Counts    0
dtype: int64

In [160]:
# save the data
ped_df.to_csv('data/Miovision_short_term_sites_data2.csv', index = False)

In [81]:
# check whether the 8 files with engine error can be opened or not. All the files only have 1KB.
df = err[err.errortype == "Excel file format cannot be determined, you must specify an engine manually."]

In [88]:
df.filename.to_list()

['Miovision_Raw_data_July2024/271485-Standard_SCU1RG_2015-10-06_0600.xlsx',
 'Miovision_Raw_data_July2024/271486-Standard_SCU1RG_2015-10-06_0600.xlsx',
 'Miovision_Raw_data_July2024/281025-MusselShoalsBikePath.xlsx',
 'Miovision_Raw_data_July2024/281450-Standard_SCU1RG_2015-11-17_0500.001.xlsx',
 'Miovision_Raw_data_July2024/281454-Standard_SCU1RG_2015-12-04_0600.xlsx',
 'Miovision_Raw_data_July2024/281465-Standard_SCU1RG_2015-12-04_0600.001.xlsx',
 'Miovision_Raw_data_July2024/281474-Standard_SCU1RG_2015-12-04_0600.xlsx',
 'Miovision_Raw_data_July2024/281492-Standard_SCU1RG_2015-12-04_0600.xlsx']

In [119]:
# confirm the 270 files don't have pedestrian count sheet
tab_names = []
raw_data_class = []
df = err[err.errortype == "Worksheet named 'Pedestrians' not found"]
for fn in df.filename:
    xlfile = archive.open(fn)
    tab_names += pd.ExcelFile(xlfile).sheet_names
    
    raw_data_class += list(pd.read_excel(xlfile, 'Raw Data')['Class'].unique())

In [121]:
np.unique(tab_names)

array(['AM Peak Class Breakdown', 'AM Weekend Peak Class Breakdown',
       'Articulated Trucks', 'Articulated Trucks and Singl...',
       'Bicycles', 'Bicycles on Road', 'Buses',
       'Buses and Single-Unit Trucks', 'Cars', 'Heavy',
       'Light Goods Vehicles', 'Lights', 'Lights and Motorcycles',
       'Midday Peak Class Breakdown', 'Midday Weekend Peak Class Br...',
       'Motorcycles', 'PM Peak Class Breakdown',
       'PM Weekend Peak Class Breakdown', 'Pivot Table', 'Raw Data',
       'Single-Unit Trucks', 'Summary', 'Total Volume Class Breakdown',
       'Vehicles'], dtype='<U31')

In [120]:
np.unique(raw_data_class)

array(['Articulated Trucks', 'Articulated Trucks and Single-Unit Trucks',
       'Bicycles', 'Bicycles on Road', 'Buses',
       'Buses and Single-Unit Trucks', 'Cars', 'Heavy',
       'Light Goods Vehicles', 'Lights', 'Lights and Motorcycles',
       'Motorcycles', 'Single-Unit Trucks', 'Vehicles'], dtype='<U41')